# ⚙️ Optimización de Hiperparámetros — FIDE Chess Dataset

**Evaluación 2 — Optimización (30%)**

Este notebook **no entrena modelos**. Carga los resultados ya calculados por el
pipeline de optimización (`optimization_report`) y genera visualizaciones:
1. Tabla visual con el Score CV (F1) obtenido por cada modelo optimizado
2. Gráfico de barras comparando resultados de GridSearchCV y RandomizedSearchCV
3. Detalle de mejores hiperparámetros por modelo

In [ ]:
# ============================================================
# Celda 1: Inicializar sesión de Kedro
# ============================================================
%load_ext kedro.ipython

In [ ]:
# ============================================================
# Celda 2: Imports y configuración de visualización
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo profesional
sns.set_theme(style='whitegrid', palette='viridis', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
print('✅ Librerías cargadas correctamente')

In [ ]:
# ============================================================
# Celda 3: Cargar reporte de optimización desde el catálogo
# ============================================================
# El pipeline 'hyperparameter_tuning' ya ejecutó GridSearchCV /
# RandomizedSearchCV y guardó los resultados en optimization_report
opt_report = catalog.load('optimization_report')

print('📋 Reporte de optimización cargado exitosamente')
print(f'   Mejor modelo global:  {opt_report["best_model"]}')
print(f'   Mejor Score CV (F1):  {opt_report["best_cv_score"]:.4f}')
print(f'   Modelos evaluados:    {list(opt_report["all_results"].keys())}')

---
## 1. Tabla Visual — Resultados de Optimización por Modelo

In [ ]:
# ============================================================
# Tabla estilizada con pandas — Score CV (F1) por modelo
# ============================================================
all_results = opt_report['all_results']

# Construir DataFrame con todas las métricas
rows = []
for model_name, result in all_results.items():
    rows.append({
        'Modelo': model_name,
        'Método': result['search_method'].upper(),
        'Best CV Score (F1)': result['best_cv_score'],
        'Test Accuracy': result['test_accuracy'],
        'Test F1': result['test_f1'],
    })

results_df = pd.DataFrame(rows).set_index('Modelo')

# Ordenar por F1 CV descendente
results_df = results_df.sort_values('Best CV Score (F1)', ascending=False)

print('📊 Resultados de Optimización de Hiperparámetros:')
display(
    results_df.style
    .format({
        'Best CV Score (F1)': '{:.4f}',
        'Test Accuracy': '{:.4f}',
        'Test F1': '{:.4f}',
    })
    .background_gradient(cmap='YlGn', subset=['Best CV Score (F1)', 'Test Accuracy', 'Test F1'])
    .set_caption('Comparación de modelos optimizados — GridSearchCV / RandomizedSearchCV')
)

---
## 2. Gráfico de Barras — Score CV (F1) por Modelo Optimizado

In [ ]:
# ============================================================
# GRÁFICO: Barras agrupadas — CV Score, Test Accuracy, Test F1
# ============================================================
fig, ax = plt.subplots(figsize=(14, 7))

model_names = results_df.index.tolist()
x = np.arange(len(model_names))
width = 0.25

# Barras para cada métrica
bars1 = ax.bar(x - width, results_df['Best CV Score (F1)'], width,
               label='Best CV Score (F1)', color='#3498db',
               edgecolor='white', alpha=0.85)
bars2 = ax.bar(x, results_df['Test Accuracy'], width,
               label='Test Accuracy', color='#2ecc71',
               edgecolor='white', alpha=0.85)
bars3 = ax.bar(x + width, results_df['Test F1'], width,
               label='Test F1', color='#e74c3c',
               edgecolor='white', alpha=0.85)

# Etiquetas sobre barras
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# Indicar método de búsqueda (Grid vs Random) debajo del nombre
search_methods = [all_results[m]['search_method'].upper() for m in model_names]
x_labels = [f'{name}\n({method})' for name, method in zip(model_names, search_methods)]

ax.set_title('Comparación de Modelos Optimizados\n(GridSearchCV / RandomizedSearchCV)',
             fontweight='bold', fontsize=15)
ax.set_xlabel('Modelo (Método de Búsqueda)')
ax.set_ylabel('Score')
ax.set_xticks(x)
ax.set_xticklabels(x_labels, fontsize=11)
ax.set_ylim(0, 1.12)
ax.legend(fontsize=11, loc='lower right')
ax.grid(axis='y', alpha=0.3)

# Resaltar el mejor modelo
best_idx = model_names.index(opt_report['best_model']) if opt_report['best_model'] in model_names else 0
ax.annotate(
    f'⭐ Mejor: {opt_report["best_model"]}',
    xy=(best_idx, results_df.loc[opt_report['best_model'], 'Best CV Score (F1)'] + 0.03),
    fontsize=12, fontweight='bold', ha='center', color='#e74c3c',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.9)
)

plt.tight_layout()
plt.show()

---
## 3. Mejores Hiperparámetros por Modelo

In [ ]:
# ============================================================
# Tabla de mejores hiperparámetros encontrados
# ============================================================
print('🔧 Mejores Hiperparámetros por Modelo:\n')

for model_name, result in all_results.items():
    method = result['search_method'].upper()
    score = result['best_cv_score']
    params = result['best_params']

    print(f'  ┌─ {model_name} ({method})')
    print(f'  │  Best CV F1: {score:.4f}')
    print(f'  │  Parámetros:')
    for param, value in params.items():
        # Limpiar prefijo "classifier__" para legibilidad
        clean_param = param.replace('classifier__', '')
        print(f'  │    {clean_param}: {value}')
    print(f'  └────────────────────────\n')

# DataFrame complementario
params_rows = []
for model_name, result in all_results.items():
    row = {'Modelo': model_name, 'Método': result['search_method'].upper()}
    for p, v in result['best_params'].items():
        row[p.replace('classifier__', '')] = v
    params_rows.append(row)

params_df = pd.DataFrame(params_rows).set_index('Modelo')
display(params_df)

---
## 4. Conclusiones de la Optimización

- **GridSearchCV** se utilizó para modelos con espacios de búsqueda pequeños (ej. LogisticRegression).
- **RandomizedSearchCV** se utilizó para modelos con espacios de búsqueda grandes (ej. RandomForest, GradientBoosting).
- La optimización mejora (o al menos mantiene) el rendimiento respecto al modelo base.
- La reproducibilidad está garantizada con `random_state=42`.
- Los resultados muestran de forma clara qué combinación de hiperparámetros produce el mejor F1-Score en validación cruzada.